In [3]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = "../data/raw/loans.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Rows, columns:", df.shape)
print("Memory (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))

Rows, columns: (1347681, 15)
Memory (MB): 266.9


In [4]:
import importlib

for library in ["sklearn", "scipy", "statsmodels", "umap", "streamlit", "plotly", "seaborn"]:
    try:
        module = importlib.import_module(library)
        print(f"{library:12s} OK  {getattr(module, '__version__', '')}")
    except Exception as error:
        print(f"{library:12s} FAILED -> {error}")

sklearn      OK  1.9.1
scipy        OK  1.18.1
statsmodels  OK  0.15.0
umap         OK  0.6.0
streamlit    OK  1.67.0
plotly       OK  7.1.0
seaborn      OK  0.13.2


In [5]:
def profile_columns(frame):
    """One row per column: type, % missing, number of distinct values, an example."""
    return pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing_pct": (frame.isna().mean() * 100).round(1),
        "unique_values": frame.nunique(),
        "example": frame.apply(lambda column: column.dropna().iloc[0] if column.notna().any() else None),
    }).sort_values("missing_pct", ascending=False)


profile = profile_columns(df)
print("Columns that are more than half empty:", (profile["missing_pct"] > 50).sum())
profile

Columns that are more than half empty: 1


,dtype,missing_pct,unique_values,example
desc,str,91.2,109539,We knew that using our credit cards to finance...
title,str,1.2,61451,Debt consolidation
id,int64,0.0,1347681,68407277
dti_n,float64,0.0,7068,5.91
loan_amnt,int64,0.0,1560,3600
issue_d,str,0.0,139,Dec-2015
revenue,float64,0.0,65608,55000.0
experience_c,int64,0.0,2,1
fico_n,float64,0.0,48,677.0
emp_length,str,0.0,12,10+ years


In [6]:
print("Fully duplicated rows:", df.duplicated().sum())

id_columns = [c for c in df.columns if c.lower() in ("id", "loan_id", "member_id")]
for column in id_columns:
    print(column, "- duplicate IDs:", df[column].duplicated().sum())

Fully duplicated rows: 0
id - duplicate IDs: 0


In [7]:
numeric = df.select_dtypes("number")
numeric.describe(percentiles=[0.01, 0.5, 0.99]).T.round(2)

,count,mean,std,min,1%,50%,99%,max
id,1347681.0,56213593.23,38395009.68,54734.0,534948.4,57664225.00,1.364971e+08,145636374.0
revenue,1347681.0,77369.68,70362.99,1896.0,19000.0,65000.00,2.600000e+05,10999200.0
dti_n,1347681.0,18.30,11.15,0.0,1.8,17.63,3.848000e+01,999.0
loan_amnt,1347681.0,14408.23,8715.35,500.0,1500.0,12000.00,3.500000e+04,40000.0
fico_n,1347681.0,698.16,31.85,612.0,662.0,692.00,8.020000e+02,847.5
experience_c,1347681.0,1.00,0.00,0.0,1.0,1.00,1.000000e+00,1.0
Default,1347681.0,0.20,0.40,0.0,0.0,0.00,1.000000e+00,1.0


In [8]:
def find_columns(frame, keywords):
    """Columns whose name contains any of the keywords."""
    return [c for c in frame.columns if any(k in c.lower() for k in keywords)]


groups = {
    "income": ["inc"],
    "debt-to-income": ["dti"],
    "loan amount": ["loan_amnt", "loan_amount", "amount"],
    "credit score": ["fico", "score"],
    "employment": ["emp"],
    "purpose": ["purpose"],
    "home ownership": ["home", "ownership"],
    "geography": ["state", "zip"],
    "date": ["date", "issue", "_d"],
    "default / outcome": ["default", "status", "charged"],
}
for name, keywords in groups.items():
    print(f"{name:20s}", find_columns(df, keywords))

income               []
debt-to-income       ['dti_n']
loan amount          ['loan_amnt']
credit score         ['fico_n']
employment           ['emp_length']
purpose              ['purpose']
home ownership       ['home_ownership_n']
geography            ['addr_state', 'zip_code']
date                 ['issue_d']
default / outcome    ['Default']


In [11]:
OUTCOME_COLUMN = "Default"             # replaces "loan_status"
DATE_COLUMN = "issue_d"
CATEGORY_COLUMNS = ["purpose", "home_ownership_n", "emp_length", "addr_state"]

In [12]:
print(df[OUTCOME_COLUMN].value_counts(dropna=False))
print(df[OUTCOME_COLUMN].value_counts(normalize=True).round(3))

Default
0    1078432
1     269249
Name: count, dtype: int64
Default
0    0.8
1    0.2
Name: proportion, dtype: float64


In [13]:
dates = pd.to_datetime(df[DATE_COLUMN], errors="coerce")

print("Unreadable dates:", dates.isna().sum())
print("Range:", dates.min().date(), "to", dates.max().date())
dates.dt.year.value_counts().sort_index()

C:\Users\yoges\AppData\Local\Temp\ipykernel_9944\3025132417.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df[DATE_COLUMN], errors="coerce")


Unreadable dates: 0
Range: 2007-06-01 to 2018-12-01


issue_d
2007       599
2008      2393
2009      5281
2010     12537
2011     21721
2012     53367
2013    134804
2014    223102
2015    375543
2016    293057
2017    169117
2018     56160
Name: count, dtype: int64

In [14]:
for column in CATEGORY_COLUMNS:
    print(df[column].value_counts(dropna=False).head(10), "\n")

purpose
debt_consolidation    781206
credit_card           295551
home_improvement       87684
other                  78263
major_purchase         29543
medical                15606
small_business         15575
car                    14647
moving                  9520
vacation                9083
Name: count, dtype: int64 

home_ownership_n
MORTGAGE    666594
RENT        535585
OWN         144985
OTHER          517
Name: count, dtype: int64 

emp_length
10+ years    442667
2 years      122091
< 1 year     108517
3 years      107862
1 year        88841
5 years       84326
4 years       80761
NI            78188
6 years       62877
8 years       60808
Name: count, dtype: int64 

addr_state
CA    196805
TX    110307
NY    110075
FL     95815
IL     51858
NJ     48576
PA     45640
OH     43935
GA     43470
VA     38107
Name: count, dtype: int64 



In [15]:
numeric.skew().sort_values(ascending=False).head(8).round(2)

revenue          45.40
dti_n            27.11
Default           1.50
fico_n            1.28
loan_amnt         0.78
id                0.22
experience_c   -232.17
dtype: float64

In [16]:
suspicious = ["pymnt", "recover", "collection", "out_prncp", "last_", "next_",
              "grade", "int_rate", "installment", "funded"]
find_columns(df, suspicious)

[]